### **Task 3: membangun model berdasarkan karakteristik peminjam, memprediksi probabilitas gagal bayar (PD), lalu memakainya untuk menghitung kerugian yang diharapkan (expected loss / EL) atas suatu pinjaman.**

**1. KERANGKA KERJA KERUGIAN YANG DIHARAPKAN (Expected Loss)**

Rumus standar credit risk yang dipakai bank:

    Expected Loss (EL) = PD  x  LGD  x  EAD

    PD  (Probability of Default) -> keluaran model (0-1), dari fitur peminjam.
    LGD (Loss Given Default)     -> proporsi eksposur yang HILANG jika gagal bayar
                                     = 1 - recovery_rate. Tugas ini memakai asumsi
                                     recovery_rate = 10%, sehingga LGD = 90% = 0.9.
    EAD (Exposure at Default)    -> jumlah uang yang berisiko saat gagal bayar.
                                     Di sini dipakai `loan_amt_outstanding` (saldo
                                     pinjaman yang belum lunas) sebagai proxy EAD,
                                     karena itulah nominal yang akan hilang jika
                                     peminjam gagal bayar pinjaman TERSEBUT.

**2. DATA**

Task_3_and_4_Loan_Data.csv berisi 10.000 baris, tidak ada nilai kosong, dengan fitur:
  - credit_lines_outstanding : jumlah pinjaman/kredit lain yang masih berjalan
  - loan_amt_outstanding     : sisa saldo pinjaman saat ini (dipakai sbg EAD)
  - total_debt_outstanding   : total utang peminjam (semua sumber)
  - income                   : pendapatan tahunan
  - years_employed           : lama bekerja (tahun)
  - fico_score               : skor kredit FICO
  - default                  : label (1 = gagal bayar, 0 = tidak) -> target model

Target sangat imbalanced ringan (18.5% default), masih cukup wajar untuk
classifier standar tanpa perlu resampling agresif, tapi kita tetap memakai
metrik yang sesuai untuk data imbalanced: AUC-ROC (bukan cuma akurasi).

**3. PILIHAN MODEL**

Dua pendekatan dilatih dan dibandingkan:
  1. Logistic Regression -> baseline yang interpretable secara industri
     (koefisien punya arti langsung sbg "credit scorecard"), cepat, dan mudah
     divalidasi oleh tim model risk/model validation (yang biasanya sangat
     concern soal interpretability & auditability model kredit).
  2. Gradient Boosted Trees (sklearn's GradientBoostingClassifier)
     -> menangkap hubungan non-linear & interaksi antar fitur (misalnya
     kombinasi FICO rendah + utang tinggi bisa jauh lebih berisiko daripada
     dijumlahkan secara linear), biasanya AUC lebih tinggi dari logistic
     regression polos, dengan trade-off interpretability lebih rendah
     (meski masih bisa dijelaskan lewat feature_importances_).

Kita bandingkan performa keduanya (AUC-ROC, akurasi, confusion matrix) lalu
pilih model dengan performa terbaik untuk fungsi akhir `predict_pd()` dan
`expected_loss()`. Model final di-fit ulang di SELURUH data (setelah evaluasi
dengan train/test split) agar prototipe memakai semua informasi yang tersedia.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report

In [2]:
RECOVERY_RATE = 0.10     # asumsi dari instruksi tugas
LGD = 1 - RECOVERY_RATE  # Loss Given Default = 90%

FEATURE_COLS = [
    "credit_lines_outstanding",
    "loan_amt_outstanding",
    "total_debt_outstanding",
    "income",
    "years_employed",
    "fico_score",
]
TARGET_COL = "default"
EAD_COL = "loan_amt_outstanding"  # proxy eksposur saat gagal bayar

## BAGIAN 1: MEMUAT DATA & MELATIH MODEL


In [3]:
def load_data(csv_path="Task_3_and_4_Loan_Data.csv"):
    df = pd.read_csv(csv_path)
    return df


def train_models(df, test_size=0.25, random_state=42):
    """
    Melatih Logistic Regression dan Gradient Boosting pada train/test split,
    lalu membandingkan performanya. Mengembalikan dict berisi kedua model
    (sudah di-fit ulang di seluruh data), scaler, dan metrik evaluasi.
    """
    X = df[FEATURE_COLS]
    y = df[TARGET_COL]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Logistic Regression (perlu fitur di-scale agar konvergen & koefisien sebanding satu sama lain)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    logreg = LogisticRegression(max_iter=1000, random_state=random_state)
    logreg.fit(X_train_scaled, y_train)
    logreg_probs = logreg.predict_proba(X_test_scaled)[:, 1]
    logreg_preds = logreg.predict(X_test_scaled)

    # Gradient Boosted Trees (tidak perlu scaling)
    gbt = GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.1, random_state=random_state
    )
    gbt.fit(X_train, y_train)
    gbt_probs = gbt.predict_proba(X_test)[:, 1]
    gbt_preds = gbt.predict(X_test)

    metrics = {
        "Logistic Regression": {
            "auc_roc": roc_auc_score(y_test, logreg_probs),
            "accuracy": accuracy_score(y_test, logreg_preds),
            "confusion_matrix": confusion_matrix(y_test, logreg_preds),
            "classification_report": classification_report(y_test, logreg_preds, digits=3),
        },
        "Gradient Boosting": {
            "auc_roc": roc_auc_score(y_test, gbt_probs),
            "accuracy": accuracy_score(y_test, gbt_preds),
            "confusion_matrix": confusion_matrix(y_test, gbt_preds),
            "classification_report": classification_report(y_test, gbt_preds, digits=3),
        },
    }

    # Fit ulang kedua model di SELURUH data untuk dipakai di produksi/prototipe
    scaler_full = StandardScaler()
    X_scaled_full = scaler_full.fit_transform(X)
    logreg_full = LogisticRegression(max_iter=1000, random_state=random_state)
    logreg_full.fit(X_scaled_full, y)

    gbt_full = GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.1, random_state=random_state
    )
    gbt_full.fit(X, y)

    best_model_name = max(metrics, key=lambda k: metrics[k]["auc_roc"])

    return {
        "metrics": metrics,
        "best_model_name": best_model_name,
        "logreg_model": logreg_full,
        "logreg_scaler": scaler_full,
        "gbt_model": gbt_full,
        "feature_importance": dict(zip(FEATURE_COLS, gbt_full.feature_importances_)),
    }

## BAGIAN 2: FUNGSI PREDIKSI PD & EXPECTED LOSS UNTUK SATU PEMINJAM


In [4]:
def predict_pd(borrower, trained, model_choice="best"):
    """
    Memprediksi probabilitas gagal bayar (PD) untuk satu peminjam.

    Parameter
    ---------
    borrower : dict
        Berisi keys: credit_lines_outstanding, loan_amt_outstanding,
        total_debt_outstanding, income, years_employed, fico_score.
    trained : dict
        Hasil dari train_models().
    model_choice : str
        "best" (default, otomatis pilih model dgn AUC tertinggi saat evaluasi),
        "logreg", atau "gbt" untuk memaksa model tertentu.

    Return
    ------
    float : probabilitas gagal bayar (0-1)
    """
    x = pd.DataFrame([{col: borrower[col] for col in FEATURE_COLS}])

    if model_choice == "best":
        model_choice = "gbt" if trained["best_model_name"] == "Gradient Boosting" else "logreg"

    if model_choice == "logreg":
        x_scaled = trained["logreg_scaler"].transform(x)
        pd_estimate = trained["logreg_model"].predict_proba(x_scaled)[0, 1]
    elif model_choice == "gbt":
        pd_estimate = trained["gbt_model"].predict_proba(x)[0, 1]
    else:
        raise ValueError("model_choice harus 'best', 'logreg', atau 'gbt'.")

    return float(pd_estimate)


def expected_loss(borrower, trained, model_choice="best", recovery_rate=RECOVERY_RATE,
                   ead_override=None):
    """
    Menghitung Expected Loss (EL) untuk satu pinjaman:

        EL = PD x LGD x EAD
        LGD = 1 - recovery_rate

    Parameter
    ---------
    borrower : dict
        Detail pinjaman (lihat predict_pd). EAD diambil dari
        borrower['loan_amt_outstanding'] kecuali `ead_override` diberikan.
    trained : dict
        Hasil dari train_models().
    recovery_rate : float
        Asumsi tingkat pemulihan (default 10% sesuai instruksi tugas).
    ead_override : float, optional
        Jika ingin memakai eksposur yang berbeda dari loan_amt_outstanding
        (misalnya committed credit line yang belum ditarik penuh).

    Return
    ------
    dict berisi 'pd', 'lgd', 'ead', dan 'expected_loss'.
    """
    pd_estimate = predict_pd(borrower, trained, model_choice=model_choice)
    lgd = 1 - recovery_rate
    ead = ead_override if ead_override is not None else borrower[EAD_COL]
    el = pd_estimate * lgd * ead

    return {
        "pd": round(pd_estimate, 4),
        "lgd": lgd,
        "ead": round(ead, 2),
        "expected_loss": round(el, 2),
    }


def portfolio_expected_loss(df_portfolio, trained, model_choice="best",
                             recovery_rate=RECOVERY_RATE):
    """
    Menghitung expected loss untuk seluruh portofolio pinjaman sekaligus
    (dipakai untuk estimasi cadangan kerugian / loss reserve tim risiko).
    """
    borrowers = df_portfolio[FEATURE_COLS].to_dict(orient="records")
    results = [
        expected_loss(b, trained, model_choice=model_choice, recovery_rate=recovery_rate)
        for b in borrowers
    ]
    result_df = df_portfolio.copy()
    result_df["predicted_pd"] = [r["pd"] for r in results]
    result_df["expected_loss"] = [r["expected_loss"] for r in results]
    return result_df

## PENGUJIAN / CONTOH PENGGUNAAN

In [9]:
if __name__ == "__main__":

    df = load_data("/content/drive/MyDrive/Forage/Quantitative Research - J.P Morgan Chase & Co/Task 3 and 4_Loan_Data.csv")
    print(f"Data dimuat: {len(df):,} baris, tingkat gagal bayar historis "
          f"{df[TARGET_COL].mean()*100:.2f}%\n")

    trained = train_models(df)

    print("=" * 72)
    print("PERBANDINGAN PERFORMA MODEL (pada test set, 25% data)")
    print("=" * 72)
    for name, m in trained["metrics"].items():
        print(f"\n--- {name} ---")
        print(f"AUC-ROC  : {m['auc_roc']:.4f}")
        print(f"Akurasi  : {m['accuracy']:.4f}")
        print(f"Confusion matrix [[TN FP],[FN TP]]:\n{m['confusion_matrix']}")
        print(m["classification_report"])

    print(f"Model dengan AUC-ROC terbaik: {trained['best_model_name']}\n")

    print("Feature importance (dari Gradient Boosting):")
    for feat, imp in sorted(trained["feature_importance"].items(), key=lambda x: -x[1]):
        print(f"  {feat:28s}: {imp:.3f}")
    print()

    # CONTOH 1: Peminjam berisiko rendah (FICO tinggi, utang rendah, income tinggi)
    low_risk_borrower = {
        "credit_lines_outstanding": 0,
        "loan_amt_outstanding": 3000,
        "total_debt_outstanding": 2000,
        "income": 90000,
        "years_employed": 8,
        "fico_score": 780,
    }
    result_low = expected_loss(low_risk_borrower, trained)
    print("Contoh 1 - Peminjam berisiko rendah:")
    print(f"  PD = {result_low['pd']:.2%}, EAD = ${result_low['ead']:,.2f}, "
          f"LGD = {result_low['lgd']:.0%}")
    print(f"  Expected Loss = ${result_low['expected_loss']:,.2f}\n")


    # CONTOH 2: Peminjam berisiko tinggi (banyak kredit berjalan, utang tinggi,
    # FICO rendah, income rendah)
    high_risk_borrower = {
        "credit_lines_outstanding": 5,
        "loan_amt_outstanding": 9000,
        "total_debt_outstanding": 15000,
        "income": 28000,
        "years_employed": 1,
        "fico_score": 560,
    }
    result_high = expected_loss(high_risk_borrower, trained)
    print("Contoh 2 - Peminjam berisiko tinggi:")
    print(f"  PD = {result_high['pd']:.2%}, EAD = ${result_high['ead']:,.2f}, "
          f"LGD = {result_high['lgd']:.0%}")
    print(f"  Expected Loss = ${result_high['expected_loss']:,.2f}\n")

    # CONTOH 3: Bandingkan PD dari kedua model untuk peminjam yang sama
    print("Contoh 3 - Perbandingan PD Logistic Regression vs Gradient Boosting "
          "(peminjam risiko tinggi):")
    pd_logreg = predict_pd(high_risk_borrower, trained, model_choice="logreg")
    pd_gbt = predict_pd(high_risk_borrower, trained, model_choice="gbt")
    print(f"  Logistic Regression PD : {pd_logreg:.2%}")
    print(f"  Gradient Boosting PD   : {pd_gbt:.2%}\n")

    # CONTOH 4: Estimasi total expected loss untuk seluruh portofolio (cadangan
    # kerugian yang disarankan untuk tim manajemen risiko)
    portfolio_result = portfolio_expected_loss(df.sample(1000, random_state=1), trained)
    total_el = portfolio_result["expected_loss"].sum()
    total_ead = portfolio_result[EAD_COL].sum()
    print("Contoh 4 - Estimasi kerugian pada sampel 1.000 pinjaman dari portofolio:")
    print(f"  Total eksposur (EAD)      : ${total_ead:,.2f}")
    print(f"  Total Expected Loss (EL)  : ${total_el:,.2f}")
    print(f"  EL sbg % dari total EAD   : {total_el/total_ead:.2%}")

Data dimuat: 10,000 baris, tingkat gagal bayar historis 18.51%

PERBANDINGAN PERFORMA MODEL (pada test set, 25% data)

--- Logistic Regression ---
AUC-ROC  : 1.0000
Akurasi  : 0.9984
Confusion matrix [[TN FP],[FN TP]]:
[[2037    0]
 [   4  459]]
              precision    recall  f1-score   support

           0      0.998     1.000     0.999      2037
           1      1.000     0.991     0.996       463

    accuracy                          0.998      2500
   macro avg      0.999     0.996     0.997      2500
weighted avg      0.998     0.998     0.998      2500


--- Gradient Boosting ---
AUC-ROC  : 0.9999
Akurasi  : 0.9960
Confusion matrix [[TN FP],[FN TP]]:
[[2032    5]
 [   5  458]]
              precision    recall  f1-score   support

           0      0.998     0.998     0.998      2037
           1      0.989     0.989     0.989       463

    accuracy                          0.996      2500
   macro avg      0.993     0.993     0.993      2500
weighted avg      0.996     0